# 🛡️ Project 03 — Spam Email Classifier

**Learning Goals**
- Understand how text is converted into numbers a model can learn from (TF-IDF)
- Train a Naive Bayes classifier — one of the simplest and most effective text classifiers
- Evaluate the model with accuracy, precision, recall, and a confusion matrix
- Interpret *why* the model makes its predictions

**Estimated Time:** 30–45 minutes

**Difficulty:** ⭐ Beginner

---
**What you'll build:** A model that reads an email message and labels it as **Ham ✅** (legitimate) or **Spam 🚨**.

## Step 1 — Install & Import Libraries

We use **scikit-learn** for the model, **pandas** for data handling, and **matplotlib/seaborn** for charts.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Libraries imported successfully!")

## Step 2 — The Dataset

We use a small inline dataset of labelled email messages. In a real project you'd load a CSV (e.g. the UCI SMS Spam Collection), but this keeps things self-contained.

Each message is labelled `"spam"` or `"ham"` (ham = legitimate email).

In [ ]:
RAW_DATA = [
    # ham
    ("ham", "Hey, are you free for lunch tomorrow?"),
    ("ham", "Please review the attached quarterly report before the meeting."),
    ("ham", "Your package has been shipped and will arrive on Friday."),
    ("ham", "Can you send me the project files when you get a chance?"),
    ("ham", "Meeting rescheduled to 3 PM — same room."),
    ("ham", "Happy birthday! Hope you have a wonderful day."),
    ("ham", "I will be out of office until Monday. For urgent matters contact Sarah."),
    ("ham", "Thanks for the update. I will take a look tonight."),
    ("ham", "The code review comments have been addressed. PR is ready for merge."),
    ("ham", "Reminder: team standup at 10 AM today."),
    ("ham", "Could you double-check the numbers in the spreadsheet?"),
    ("ham", "The flight is confirmed. Boarding at gate 12."),
    ("ham", "Great work on the presentation! The client loved it."),
    ("ham", "Please submit your timesheets by end of day Friday."),
    ("ham", "I found a great recipe — shall I send it over?"),
    ("ham", "Your doctor appointment is confirmed for Tuesday at 2 PM."),
    ("ham", "Library book renewal successful. Due date extended by 3 weeks."),
    ("ham", "Call me when you land. Safe travels!"),
    ("ham", "The patch has been deployed to production successfully."),
    ("ham", "Budget approval received. You can proceed with the purchase."),
    ("ham", "See you at the conference next week."),
    ("ham", "The invoice for March has been processed."),
    ("ham", "Jenkins build #472 passed all tests."),
    ("ham", "Dinner reservation confirmed for Saturday at 7 PM."),
    ("ham", "The kids recital starts at 6 PM. Bring the camera!"),
    ("ham", "Annual performance reviews begin next week. Schedule your 1-on-1s."),
    # spam
    ("spam", "CONGRATULATIONS! You have won a $1000 Walmart gift card. Click here NOW!"),
    ("spam", "Earn $500 per day working from home. No experience required. Sign up free!"),
    ("spam", "URGENT: Your account has been compromised. Verify your password immediately."),
    ("spam", "Limited time offer! Buy 1 get 3 FREE. Order before midnight!"),
    ("spam", "You have been selected for a special loan offer. Apply now, no credit check!"),
    ("spam", "HOT SINGLES in your area want to meet you. Click to view profiles!"),
    ("spam", "FREE iPhone 15 Pro! You are our lucky winner. Claim your prize today."),
    ("spam", "Make money fast! Proven system earns $10,000 monthly. Join thousands!"),
    ("spam", "Your PayPal account is suspended. Log in immediately to restore access."),
    ("spam", "Lose 20 lbs in 2 weeks! Miracle pill doctors don't want you to know about."),
    ("spam", "Cheap medication online — no prescription needed. Huge discounts!"),
    ("spam", "Double your investment in 24 hours. 100% guaranteed returns. Act now!"),
    ("spam", "You owe a tax refund. Click the link to claim $3,200 from the IRS."),
    ("spam", "Enlarge and enhance — guaranteed results or your money back!"),
    ("spam", "Final notice: your subscription expires today. Renew for FREE access!"),
    ("spam", "Click HERE to unsubscribe from our mailing list. Or win a cruise!"),
    ("spam", "Exclusive VIP offer: Casino bonus $500 FREE. Register in 60 seconds!"),
    ("spam", "Your Microsoft account requires immediate verification. Click to confirm."),
    ("spam", "Stock tip: buy XYZ shares now before the price explodes tomorrow!"),
    ("spam", "Congratulations winner! Collect your Amazon voucher worth $250 now!"),
    ("spam", "Act fast — only 3 spots left in our wealth secrets webinar!"),
    ("spam", "You have a pending wire transfer of $8,500. Confirm your details."),
    ("spam", "Surprise! Click to reveal your exclusive reward from our survey."),
    ("spam", "SALE 90% off designer brands. Today only. Shop now!"),
]

df = pd.DataFrame(RAW_DATA, columns=["label", "message"])
df["is_spam"] = (df["label"] == "spam").astype(int)

print(f"Total messages : {len(df)}")
print(f"Ham (legit)    : {(df.is_spam == 0).sum()}")
print(f"Spam           : {(df.is_spam == 1).sum()}")
df.head(6)

## Step 3 — Explore the Data

Before modelling, let's look at the class distribution and some sample messages.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
counts = df["label"].value_counts()
axes[0].bar(counts.index, counts.values, color=["steelblue", "tomato"], edgecolor="white")
axes[0].set_title("Class Distribution")
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 0.2, str(v), ha="center", fontweight="bold")

# Message length distribution
df["msg_len"] = df["message"].str.len()
for label, colour in [("ham", "steelblue"), ("spam", "tomato")]:
    axes[1].hist(df[df.label == label]["msg_len"], bins=15, alpha=0.6, label=label, color=colour)
axes[1].set_title("Message Length Distribution")
axes[1].set_xlabel("Characters")
axes[1].set_ylabel("Frequency")
axes[1].legend()

plt.tight_layout()
plt.savefig("eda_plots.png", dpi=110)
plt.show()
print("EDA charts saved.")

## Step 4 — Train / Test Split

We hold out 20 % of messages for testing. `stratify=df["is_spam"]` ensures both splits have similar spam/ham ratios.

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df["message"], df["is_spam"],
    test_size=0.2, random_state=42, stratify=df["is_spam"]
)

print(f"Training samples : {len(X_train_raw)}")
print(f"Test samples     : {len(X_test_raw)}")

## Step 5 — TF-IDF Feature Extraction

**How it works:**
- **TF (Term Frequency):** How often does a word appear in *this* message?
- **IDF (Inverse Document Frequency):** How rare is the word across *all* messages?
- **TF-IDF = TF × IDF:** Words that are common in one message but rare overall get a high score (e.g. "free", "prize", "click").

We also use **bigrams** (`ngram_range=(1,2)`) — two-word phrases like *"click here"* or *"credit check"* can be more informative than individual words.

In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=500,
    ngram_range=(1, 2),
)

# IMPORTANT: fit_transform on train data only → learn vocabulary from training set
X_train = vectorizer.fit_transform(X_train_raw)
# transform on test data → apply the SAME vocabulary (no data leakage)
X_test = vectorizer.transform(X_test_raw)

print(f"Feature matrix shape (train) : {X_train.shape}")
print(f"  → {X_train.shape[0]} messages × {X_train.shape[1]} TF-IDF features")

## Step 6 — Train the Naive Bayes Model

**Why Naive Bayes?**
It assumes each word independently contributes to the class probability — "naive" but surprisingly effective for text!

The `alpha=1.0` parameter is **Laplace smoothing** — it prevents zero probabilities for words not seen during training.

In [ ]:
model = MultinomialNB(alpha=1.0)
model.fit(X_train, y_train)
print("Model trained! Classes:", model.classes_)  # [0=ham, 1=spam]

## Step 7 — Evaluate the Model

In [ ]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Accuracy: {acc*100:.1f}%\n")
print(classification_report(y_test, y_pred, target_names=["Ham", "Spam"]))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Ham", "Spam"], yticklabels=["Ham", "Spam"],
            ax=ax, linewidths=0.5)
ax.set_title("Confusion Matrix", fontsize=13)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=110)
plt.show()
print("\nHow to read this:")
print(f"  True Negatives  (Ham  → Ham)  : {cm[0,0]}")
print(f"  False Positives (Ham  → Spam) : {cm[0,1]}  ← legit emails wrongly flagged")
print(f"  False Negatives (Spam → Ham)  : {cm[1,0]}  ← spam that slipped through")
print(f"  True Positives  (Spam → Spam) : {cm[1,1]}")

## Step 8 — Interpret the Model

Which words does the model consider most "spammy"? We compare the log-probabilities for each word in the spam class vs. the ham class.

In [ ]:
feature_names = vectorizer.get_feature_names_out()
log_prob_diff  = model.feature_log_prob_[1] - model.feature_log_prob_[0]

top_n = 15
top_spam_idx = np.argsort(log_prob_diff)[-top_n:][::-1]
top_ham_idx  = np.argsort(log_prob_diff)[:top_n]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top spam words
axes[0].barh(feature_names[top_spam_idx][::-1], log_prob_diff[top_spam_idx][::-1], color="tomato")
axes[0].set_title(f"Top {top_n} SPAM indicators", fontsize=12)
axes[0].set_xlabel("Log-prob diff (spam − ham)")

# Top ham words
axes[1].barh(feature_names[top_ham_idx], log_prob_diff[top_ham_idx], color="steelblue")
axes[1].set_title(f"Top {top_n} HAM indicators", fontsize=12)
axes[1].set_xlabel("Log-prob diff (spam − ham)")

plt.tight_layout()
plt.savefig("feature_importance.png", dpi=110)
plt.show()

## Step 9 — Live Predictions

Let's classify some brand-new messages the model has never seen.

In [ ]:
new_messages = [
    "URGENT: Your bank account has been locked. Verify now to restore access!",
    "Hey, just checking if you got my last email about the project deadline.",
    "Congratulations! You won a FREE vacation. Click to claim your prize!",
    "Please find the attached invoice for your review.",
    "Make $10,000 a week from home — guaranteed, no experience needed!",
    "Reminder: performance review scheduled for next Thursday at 10 AM.",
]

X_new = vectorizer.transform(new_messages)
preds  = model.predict(X_new)
probas = model.predict_proba(X_new)

results = pd.DataFrame({
    "Message"    : [m[:60] + "…" if len(m) > 60 else m for m in new_messages],
    "Prediction" : ["🚨 SPAM" if p == 1 else "✅ Ham" for p in preds],
    "Spam %"     : [f"{p[1]*100:.1f}%" for p in probas],
})

results

## 🧪 Try It Yourself!

Modify the cell below with your own messages and see what the model predicts. Try borderline cases like:
- `"Your order has been confirmed. Click to track."`
- `"Free coffee at the office today!"`
- `"Exclusive discount for premium members only."`

In [ ]:
# ✏️  Edit these messages and re-run!
my_messages = [
    "Your free trial expires tomorrow. Upgrade now!",
    "Can we reschedule our call to Thursday?",
]

X_mine = vectorizer.transform(my_messages)
p = model.predict(X_mine)
prob = model.predict_proba(X_mine)

for msg, pred, pr in zip(my_messages, p, prob):
    label = "🚨 SPAM" if pred == 1 else "✅ Ham"
    print(f"{label}  ({pr[1]*100:.1f}% spam)  →  {msg}")

## 📚 Summary — What You Learned

| Concept | Plain-English Explanation |
|---|---|
| TF-IDF | Turns words into numbers, giving higher scores to rare but meaningful words |
| Bigrams | Two-word phrases (e.g. "click here") can be more informative than single words |
| Multinomial Naive Bayes | Probabilistic classifier that works well for word-count-like features |
| Laplace Smoothing | Prevents zero probabilities for unseen words |
| Train/Test Split | Hold-out set ensures we measure generalisation, not memorisation |
| Precision | Of all messages flagged as spam, how many actually were? |
| Recall | Of all real spam messages, how many did we catch? |
| Confusion Matrix | Grid showing all four prediction outcomes (TP, TN, FP, FN) |
| Feature Importance | Log-prob difference reveals which words the model considers most spammy |

**Next Steps:**
- Try the [UCI SMS Spam Collection dataset](https://archive.ics.uci.edu/ml/datasets/sms+spam+collection) (~5,500 real messages) for much stronger results
- Experiment with `LogisticRegression` instead of Naive Bayes
- Try `CountVectorizer` vs `TfidfVectorizer` and compare accuracy